In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from surrogate_models import load_neutron_stars


In [ ]:
FEATURES= ['rhoc', 'lambda', 'beta']
TARGETS = ['M', 'D']

In [ ]:
df: pd.DataFrame = load_neutron_stars()[FEATURES + TARGETS]
print(df.shape), df.info()

In [ ]:
df.describe().T

In [ ]:
assert (df['rhoc'] > 0).all(), "All rhoc values must be positive"
assert df.isna().sum().sum() == 0, "There are missing values in the dataset"
assert df.duplicated(subset=FEATURES).sum() == 0, \
    "There are duplicate rows in the dataset"

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, col in zip(axes[0], FEATURES+ TARGETS):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
for ax, col in zip(axes[1], FEATURES + TARGETS):
    sns.histplot(np.log10(df[col].clip(lower=1e-30)), kde=True, ax=ax)
    ax.set_title(f"log10({col})")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharey='row')
for i, y in enumerate(TARGETS):
    for j, x in enumerate(FEATURES):
        hue = [c for c in FEATURES if c != x][0]
        sns.scatterplot(
            data=df,
            x=x,
            y=y,
            hue=hue,
            palette="viridis",
            s=8,
            ax=axes[i, j],
            legend=(j == 2),
        )
        if x == "rhoc":
            axes[i, j].set_xscale("log")
plt.tight_layout()

In [ ]:
sns.pairplot(
    df.sample(min(2000, len(df))),
    vars=FEATURES + TARGETS,
    plot_kws={"s": 6, "alpha": 0.4},
)
print(df.corr(method="spearman"))  # Spearman > Pearson: catches monotonic non-linear


In [ ]:
sns.heatmap(
    df.pivot_table(index="beta", columns="lambda", values="rhoc", aggfunc="count"),
    annot=True,
    fmt=".0f",
)


In [ ]:
X, y = df[FEATURES].copy(), df[TARGETS]
X["log_rhoc"] = np.log10(X["rhoc"])  # cheap feature-engineering probe

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "linear": make_pipeline(StandardScaler(), LinearRegression()),
    "poly3": make_pipeline(
        PolynomialFeatures(3, include_bias=False), StandardScaler(), LinearRegression()
    ),
    "rf": RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1),
}
for name, m in models.items():
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)
    for k, t in enumerate(TARGETS):
        print(
            f"{name:8s} {t}: R²={r2_score(yte[t], pred[:, k]):.4f}  "
            f"MAE={mean_absolute_error(yte[t], pred[:, k]):.4g}"
        )
